# Lesson 06 · Beta-binomial regression

**Source:** *Introduction to Empirical Bayes: Examples from Baseball Statistics*, David Robinson (2017), Chapter 7 (pp. 56–64)

**Question:** Better hitters get more at-bats. If we shrink everyone toward one global average, are we systematically overrating players with short careers?

**Goal:** Rewrite the beta prior in terms of a mean μ and a dispersion σ, let μ depend on log(AB), fit it by maximum likelihood, and shrink each player toward his own prior.

**Contents**

1. Setup
2. The problem: AB and average are related
3. Reparameterizing the beta: μ and σ
4. The model and its likelihood
5. Fit it (step 1)
6. Shrink toward the trend (step 2)
7. Exercises

**Tags:** `beta-binomial-regression` `confounding` `reparameterization` `maximum-likelihood` `shrinkage` `prior-depends-on-covariates`

**Before you start:** Lessons 02 and 05. This lesson writes a general fitting function you'll reuse in Lessons 07 and 10.

*How this notebook works:* each **Your turn** prompt is followed by an empty cell for your code. **Check** boxes give values to compare against; they come from the book, and your numbers can differ slightly because the Lahman data now runs through more recent seasons. **Reflect** prompts are for short written answers. Every prompt is numbered *lesson.section.question* (for example **2.3.1**), so you can refer to it; empty code cells and answer cells carry the same number.

In [ ]:
# --- Setup: run this cell first ------------------------------------------
# Windows + conda: put the environment's DLL folders on PATH, so plotting
# can't crash the kernel when Jupyter wasn't started from the activated env.
import os, sys
for sub in ["Library\\mingw-w64\\bin", "Library\\usr\\bin", "Library\\bin", "Scripts", "bin"]:
    p = os.path.join(sys.prefix, sub)
    if os.path.isdir(p) and p not in os.environ["PATH"]:
        os.environ["PATH"] = p + os.pathsep + os.environ["PATH"]

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy
from scipy import stats, optimize, special

import eb_data   # helpers in this folder: loading the Lahman data, saving results

rng = np.random.default_rng(2017)
pd.set_option("display.precision", 4)
plt.rcParams["figure.figsize"] = (7, 4)
print(f"numpy {np.__version__} | pandas {pd.__version__} | scipy {scipy.__version__}")

career = eb_data.career(pitcher_rule=3)
alpha0, beta0 = eb_data.load_prior()
print(f"{len(career):,} players | prior alpha0 = {alpha0:.1f}, beta0 = {beta0:.1f}")

## 1. Setup

**6.1.1 · Your turn.** Add `eb_estimate`, `alpha1` and `beta1` using the single prior, as before.

In [ ]:
# 6.1.1 · Your code here

## 2. The problem: AB and average are related

**6.2.1 · Your turn.** Plot raw `average` against `AB` (log x-axis) for players with AB ≥ 10. Add a smooth trend: a binned median, or `np.polyfit` on log AB.

In [ ]:
# 6.2.1 · Your code here

**6.2.2 · Your turn.** What's the median raw average for players with 5–20 AB? What does single-prior shrinkage turn them into? Plot `eb_estimate` against AB with the prior mean as a horizontal line.

In [ ]:
# 6.2.2 · Your code here

> **Check:** the median raw average for 5–20 AB is about 0.167, far below the prior mean of about 0.26.

**6.2.3 · Reflect.** This relationship isn't measurement noise: why does it exist? In which direction does it bias the single-prior estimates, and for which players?

*6.2.3 · Your answer:*

## 3. Reparameterizing the beta: μ and σ

Write the beta in terms of its mean $\mu$ and a dispersion $\sigma$:

$$\mu = \frac{\alpha}{\alpha+\beta}, \quad \sigma = \frac{1}{\alpha+\beta} \qquad\Longleftrightarrow\qquad \alpha = \frac{\mu}{\sigma}, \quad \beta = \frac{1-\mu}{\sigma}.$$

Small $\sigma$ means a tight, informative prior.

**6.3.1 · Your turn.** Write `to_mu_sigma(alpha, beta)` and `to_alpha_beta(mu, sigma)`. Convert your α₀, β₀ and back.

In [ ]:
# 6.3.1 · Your code here

## 4. The model and its likelihood

Now let the prior mean depend on at-bats:

$$\mu_i = \mu_0 + \mu_{AB}\,\log(AB_i), \qquad p_i \sim \text{Beta}\!\left(\frac{\mu_i}{\sigma_0}, \frac{1-\mu_i}{\sigma_0}\right), \qquad H_i \sim \text{Binomial}(AB_i, p_i).$$

This is **beta-binomial regression**. The book fits it with R's `gamlss`. We'll write the likelihood ourselves. Write it generally, with a design matrix `X` (a column of ones plus predictors) and coefficients `b`, so that $\mu = Xb$. Then Lesson 07 can add predictors without new code.

**6.4.1 · Your turn.** Write `bb_reg_neg_log_lik(params, H, AB, X)`, where `params = [*b, log_sigma]`:
- `mu = X @ b`, clipped to (1e-6, 1 − 1e-6) so that bad steps don't crash the optimizer;
- `alpha = mu / sigma`, `beta = (1 - mu) / sigma`;
- return `-stats.betabinom.logpmf(H, AB, alpha, beta).sum()`.

In [ ]:
# 6.4.1 · Your code here

## 5. Fit it (step 1)

**6.5.1 · Your turn.** Write `fit_bb_regression(H, AB, X, start=None)` that calls `optimize.minimize` and returns the coefficients, σ, and (optionally) standard errors from the inverse Hessian. Fit it with `X = [1, log(AB)]` on all players.

Good starting values help: the intercept near 0.15, slope near 0.01, `log_sigma` near −6. Try `method="L-BFGS-B"`, and `"Nelder-Mead"` if it struggles.

In [ ]:
# 6.5.1 · Your code here

> **Check:** μ₀ ≈ 0.143, μ_AB ≈ 0.0153, σ₀ ≈ exp(−6.29) ≈ 0.0019.

**6.5.2 · Your turn.** Plot the prior density implied for players with AB = 1, 10, 100, 1,000 and 10,000.

In [ ]:
# 6.5.2 · Your code here

## 6. Shrink toward the trend (step 2)

**6.6.1 · Your turn.** Give each player his own prior (`alpha0_i`, `beta0_i` from his μᵢ), update with H and AB, and store `new_eb`.

In [ ]:
# 6.6.1 · Your code here

**6.6.2 · Your turn.** Compare `eb_estimate` with `new_eb`: scatter one against the other, and plot both against AB (log x) with the fitted trend line.

In [ ]:
# 6.6.2 · Your code here

**6.6.3 · Reflect.** The book points out that AB is only known at the *end* of a career. What does that mean for using this model on a rookie? Which later results (credible intervals, PEPs, A/B tests) change with the new prior?

*6.6.3 · Your answer:*

**6.6.4 · Your turn.** Save your coefficients: `eb_data.save_result("bb_reg_logAB", {...})`.

In [ ]:
# 6.6.4 · Your code here

## 7. Exercises

**6.7.1 · Your turn.** Refit with `log10(AB)` instead of `log(AB)`. Show that the fitted priors are identical and only the coefficient rescales. By what factor?

In [ ]:
# 6.7.1 · Your code here

**6.7.2 · Your turn.** Use a *logit* link instead: `mu = expit(X @ b)` (`special.expit`). It keeps μ inside (0, 1) without clipping. Do the shrunken estimates change much?

In [ ]:
# 6.7.2 · Your code here

**6.7.3 · Reflect.** **Card connection.** Suppose you model the sale prices of trading cards, as notebook 07 of Cardprice (a separate card-price project, not included in this repository) does. A card with only a few recorded sales has a noisy price estimate, so each card's price effect is shrunk toward what its *traits* predict (its age, its *rarity*, meaning the printed tier saying how scarce it is, and so on), not toward one global mean. How is that the same idea as this lesson? What plays the role of log(AB)?

*6.7.3 · Your answer:*

## References

- *Introduction to Empirical Bayes: Examples from Baseball Statistics*, David Robinson (2017), Chapter 7.
- Ferrari & Cribari-Neto (2004), "Beta regression for modeling rates and proportions", *J. Applied Statistics* 31(7).
- Stasinopoulos & Rigby, GAMLSS: https://www.gamlss.com/
- Gelman, Hill & Vehtari, *Regression and Other Stories*, Ch. 15 (other GLMs).